## 1. Import modules and data

In [1]:
!pip install numpy-financial

# Import common libraries
import pandas as pd
import os
from google.colab import drive
from google.colab import auth
import gspread
from google.auth import default
from collections import defaultdict
import numpy as np
import re
from scipy import stats
import numpy_financial as npf

### Set up Google Drive

# Mount Google Drive
drive.mount('/content/drive')

# Path to datag
project_path = '/content/drive/MyDrive/Data and Polling/Current Projects/HBF/Data/WAS_data'
os.chdir(project_path)

#Set up access to google sheet
from google.colab import auth
from google.auth import default
from google.auth.transport.requests import AuthorizedSession
import gspread

# Authenticate
auth.authenticate_user()

# Get credentials with the right scopes
creds, _ = default(scopes=[    'https://www.googleapis.com/auth/drive', 'https://www.googleapis.com/auth/spreadsheets'])

# Build gspread client correctly
gc = gspread.Client(auth=creds)
gc.session = AuthorizedSession(creds)

#Function to load google datasets
def load_data_from_google_sheet (sheet_url, worksheet_name):
  sh = gc.open_by_url(sheet_url)
  worksheet = sh.worksheet(worksheet_name)
  rows = worksheet.get_all_values()
  df = pd.DataFrame.from_records(rows)
  new_header = df.iloc[0] #grab the first row for the header
  df = df[1:] #take the data less the header row
  df.columns = new_header #set the header row as the df header
  return df



Mounted at /content/drive


2. Create inflator functions for nominal house prices, wealth and income that provide multipliers between any two years between 1999 and 2005

In [8]:
os.chdir(project_path)

#import data
house_price_time_series = load_data_from_google_sheet("https://docs.google.com/spreadsheets/d/1ogjzwRD2VK5Sb3a3D_fnf2riJk1a1JcI7eIRUGGNvtY/edit?usp=sharing","Data")
wealth_time_series = load_data_from_google_sheet("https://docs.google.com/spreadsheets/d/1PljQJPvxMXAZHmxWMLnC-dCsw2dyuLfFITgIkvTUXYE/edit?usp=sharing","Data")
income_time_series = load_data_from_google_sheet("https://docs.google.com/spreadsheets/d/1CYwDR0YBMQJvFwywN8IYED8_3xcwMplvLDGPRJijJLk/edit?usp=sharing","Data")

import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

#make year column the index
house_price_time_series = house_price_time_series.set_index("Year")
wealth_time_series = wealth_time_series.set_index("Year")
income_time_series = income_time_series.set_index("Year")


def fill_missing_values(series):
    # Ensure the index is treated as integers
    series.index = series.index.astype(int)

    # Convert series index to a DataFrame for regression
    df = series.dropna().reset_index()
    df.columns = ['Year', 'Value']

    # Reshape the data for regression
    X_train = df['Year'].values.reshape(-1, 1)
    y_train = df['Value'].values

    # Train the linear regression model
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Extend years to include 2009, 2010, and 2026
    all_years = np.arange(2009, 2027)
    missing_years = [year for year in all_years if year not in series.index]

    # Predict missing values
    if missing_years:
        X_pred = np.array(missing_years).reshape(-1, 1)
        y_pred = model.predict(X_pred)

        # Fill missing values
        for year, value in zip(missing_years, y_pred):
            series.loc[year] = value

    return series


fill_missing_values(house_price_time_series)
fill_missing_values(wealth_time_series)
fill_missing_values(income_time_series)

#print(house_price_time_series)
#print(wealth_time_series)
#print(income_time_series)

#function that finds multilier between a base year and a given year for a time series
def find_increase_from_year(year_survey, year_want, time_series):

  time_series["Value"] = time_series["Value"].astype(float)
  year_want_value = time_series.loc[year_want, 'Value']
  year_survey_value = time_series.loc[year_survey, 'Value']
  increase = year_want_value / year_survey_value

  return increase

print(find_increase_from_year(2021, 2026, house_price_time_series))
print(find_increase_from_year(2021, 2026, wealth_time_series))
print(find_increase_from_year(2021, 2026, income_time_series))


1.1843884210872047
1.1156046072804453
1.1959253853528196


**Don't need to run**  -  This is just to explore key variables from WAS

In [9]:
os.chdir(project_path)

#assign survey year
survey_year = 2021

#create dictionary
year_round_lookup_dict = {2021: "round_8", 2019: "round_7", 2017: "round_6"}

#read in person and hh level WAS
person_WAS = pd.read_csv(f"was_{year_round_lookup_dict[survey_year]}_person.tab", sep="\t")
hh_WAS = pd.read_csv(f"was_{year_round_lookup_dict[survey_year]}_hhold.tab", sep="\t")

# Load variable names
variable_name_sheet = load_data_from_google_sheet(
    "https://docs.google.com/spreadsheets/d/1WHPnzAXqIiGfw2wricmdsWYnSKKMRrnMQEN1NQhdwWM/edit?usp=sharing", "Data"
    )
variable_name_sheet = variable_name_sheet[variable_name_sheet["Year"].astype(int) == survey_year]
variable_map = variable_name_sheet.set_index("Variable name")["Round variable name"].to_dict()

# Retrieve and assign values
person_ID_var = variable_map["Person_ID"]
hh_ID_var = variable_map["Household_ID"]
hh_type_var = variable_map["Household_type"]
hh_tenure_var = variable_map["Tenure_type"]
non_dpndt_chld_var = variable_map["Non_dependent_child_flag"]
person_ttl_wlth_var = variable_map["Person_total_wealth"]
dpndt_chld_var = variable_map["Dependent_child_flag"]
person_ttl_pay_var = variable_map["Person_total_pay"]
emplmnt_situ_var = variable_map["Employment_situation"]
age_group_var = variable_map["Age_grouping"]
person_weight_var = variable_map["Person_weight"]
region_var = variable_map["Region"]
own_property = variable_map["Own_property"]
pension_wealth = variable_map["Pension_wealth"]

# Merge household data to person data
person_situ_df = person_WAS.merge(
    hh_WAS[[hh_ID_var, hh_type_var, hh_tenure_var]], on=hh_ID_var, how="left"
)

# List of variables to analyse
variables_to_analyse = [hh_ID_var, hh_tenure_var, non_dpndt_chld_var, dpndt_chld_var, emplmnt_situ_var]

# count weighted unique varibles
unique_value_sums = {
    var: person_situ_df.groupby(var)[person_weight_var].sum().reset_index()
    for var in variables_to_analyse
}

for var in unique_value_sums:
    unique_value_sums[var].columns = [var, "Sum of person_weight_var"]

for var, df in unique_value_sums.items():
    print(f"\nSum of person_weight_var for unique values in {var}:")
    print(df)


/tmp/ipykernel_4549/990354872.py:10: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,24,25,26,27,28,29,270,272,273,568,605,1143,3806,3815,3816,3817,3818,3819,3820,3821,3822,3823,3824,3825) have mixed types. Specify dtype option on import or set low_memory=False.
  person_WAS = pd.read_csv(f"was_{year_round_lookup_dict[survey_year]}_person.tab", sep="\t")



Sum of person_weight_var for unique values in CASER8:
       CASER8  Sum of person_weight_var
0           1                355.104284
1           2                350.482657
2           3               5453.800549
3           4                589.446223
4           5               2822.627853
...       ...                       ...
15123   15128               1005.078031
15124   15129               9490.087909
15125   15130                724.815843
15126   15131              45310.181803
15127   15132              13970.241625

[15128 rows x 2 columns]

Sum of person_weight_var for unique values in ten1r8_i:
   ten1r8_i  Sum of person_weight_var
0         1              1.665755e+07
1         2              2.226845e+07
2         3              3.083931e+05
3         4              2.393860e+07
4         5              7.322313e+05
5         6              2.248860e+02

Sum of person_weight_var for unique values in IsNDepR8:
   IsNDepR8  Sum of person_weight_var
0         1          

3. Create functions that calculate income tax and NI for any gross annual salary

In [10]:
os.chdir(project_path)

#create net income after taxes and living costs
def calculate_income_tax(gross_income):
    """
Calculate the total income tax for a given gross income in the UK for the 2024/2025 tax year.
    """
    # Define tax thresholds and rates
    personal_allowance = 12570  # Tax-free allowance
    basic_rate_threshold = 50270
    higher_rate_threshold = 125140

    # Initialize tax amount
    tax = 0

    # Adjust personal allowance for high earners
    if gross_income > 100000:
      personal_allowance -= (gross_income - 100000) / 2
      if personal_allowance < 0:
          personal_allowance = 0

    # Calculate taxable income
    taxable_income = max(0, gross_income - personal_allowance)

    # Apply tax rates
    if taxable_income <= 0:
        tax = 0
    elif taxable_income <= (basic_rate_threshold - personal_allowance):
        tax = taxable_income * 0.20
    elif taxable_income <= (higher_rate_threshold - personal_allowance):
        tax = (basic_rate_threshold - personal_allowance) * 0.20 + (taxable_income - (basic_rate_threshold - personal_allowance)) * 0.40
    else:
        tax = (basic_rate_threshold - personal_allowance) * 0.20 + (higher_rate_threshold - basic_rate_threshold) * 0.40 + (taxable_income - (higher_rate_threshold - personal_allowance)) * 0.45

    return tax

print(calculate_income_tax(70000))

def calculate_national_insurance(gross_income):
    """
    Calculate the total National Insurance (NI) contributions for a given gross income in the UK for the 2024/2025 tax year.
    """
    # Define NI thresholds and rates
    primary_threshold = 12570
    upper_earnings_limit = 50270
    ni_rate_main = 0.08
    ni_rate_additional = 0.02

    # Initialize NI contribution
    ni_contribution = 0

    # Calculate NI contributions based on income thresholds
    if gross_income <= primary_threshold:
        ni_contribution = 0
    elif gross_income <= upper_earnings_limit:
        ni_contribution = (gross_income - primary_threshold) * ni_rate_main
    else:
        ni_contribution = ((upper_earnings_limit - primary_threshold) * ni_rate_main) + \
                          ((gross_income - upper_earnings_limit) * ni_rate_additional)

    return ni_contribution

print(calculate_national_insurance(70000))



15432.0
3410.6


4. Single people who are renting or living rent free

In [24]:
os.chdir(project_path)

def singles_who_can_buy(survey_year,year,deposit_size,income_multiplier,discount,london_discount,purchase_costs,interest_rate,mortgage_term, wiggle_room, apply_sdlt=True):



    ### 1 ###
    # Create empty dictionary to save intermediate values
    intermediates = {}

    ### 2 ###
    # Create dictionary so that I can select the right WAS dataset using year
    year_round_lookup_dict = {2021: "round_8", 2019: "round_7", 2017: "round_6", 2015: "round_5"}

    if survey_year not in year_round_lookup_dict:
        return "WAS dataset not available for this year. Change survey year"

    # Load the person and household level WAS data for survey year chosen
    person_WAS = pd.read_csv(f"was_{year_round_lookup_dict[survey_year]}_person.tab", sep="\t")
    hh_WAS = pd.read_csv(f"was_{year_round_lookup_dict[survey_year]}_hhold.tab", sep="\t")

    # Create a DataFrame with just the column names in a column and export to csv
    hh_WAS_columns = pd.DataFrame(hh_WAS.columns, columns=["column_name"])
    hh_WAS_columns.to_csv("hh_WAS_columns.csv", index=False)

    person_WAS_columns = pd.DataFrame(person_WAS.columns, columns=["column_name"])
    person_WAS_columns.to_csv("person_WAS_columns.csv", index=False)

    # Load pre-prepared dataframe that acts as a key for variable names used in code and the column headings from WAS for each year, and then reduces to the year required
    variable_name_sheet = load_data_from_google_sheet(
        "https://docs.google.com/spreadsheets/d/1WHPnzAXqIiGfw2wricmdsWYnSKKMRrnMQEN1NQhdwWM/edit?usp=sharing", "Data"
    )
    variable_name_sheet = variable_name_sheet[variable_name_sheet["Year"].astype(int) == survey_year]
    variable_map = variable_name_sheet.set_index("Variable name")["Round variable name"].to_dict()

    # Create specific python variables
    person_ID_var = variable_map["Person_ID"]
    hh_ID_var = variable_map["Household_ID"]
    hh_type_var = variable_map["Household_type"]
    hh_tenure_var = variable_map["Tenure_type"]
    non_dpndt_chld_var = variable_map["Non_dependent_child_flag"]
    person_ttl_wlth_var = variable_map["Person_total_wealth"]
    dpndt_chld_var = variable_map["Dependent_child_flag"]
    person_ttl_pay_var = variable_map["Person_total_pay"]
    emplmnt_situ_var = variable_map["Employment_situation"]
    age_group_var = variable_map["Age_grouping"]
    person_weight_var = variable_map["Person_weight"]
    region_var = variable_map["Region"]
    own_property = variable_map["Own_property"]
    pension_wealth = variable_map["Pension_wealth"]

       # *** ADD THIS BLOCK HERE TO DROP UNECESSARY COLUMNS***
    cols_to_keep = [person_ID_var, hh_ID_var, hh_type_var, hh_tenure_var,
                    non_dpndt_chld_var, person_ttl_wlth_var, dpndt_chld_var,
                    person_ttl_pay_var, emplmnt_situ_var, age_group_var,
                    person_weight_var, region_var, own_property, pension_wealth]
    person_WAS = person_WAS[[c for c in cols_to_keep if c in person_WAS.columns]]
    hh_WAS = hh_WAS[[hh_ID_var, hh_type_var, hh_tenure_var]]
    # *** END OF ADDED BLOCK ***


    ### 3 ###
    #count and record lenth of dataframes
    hh_WAS_length = len(hh_WAS)
    person_WAS_length = len(person_WAS)
    intermediates["original_hh_WAS_length"] = hh_WAS_length
    intermediates["original_person_WAS_length"] = person_WAS_length

    #count the number of unique values in the column hh_ID_var
    unique_hh_ids_in_hh_WAS = hh_WAS[hh_ID_var].nunique()
    intermediates["unique hh ids in original hh WAS"] = unique_hh_ids_in_hh_WAS
    unique_hh_ids_in_person_WAS = person_WAS[hh_ID_var].nunique()
    intermediates["unique hh ids in original person WAS"] = unique_hh_ids_in_person_WAS
    unique_person_ids = person_WAS[person_ID_var].nunique()
    intermediates["unique person ids in original person WAS"] = unique_person_ids


###############  THIS MIGHT BE THE PROBLEM
    #Drop the NA variables in 2015 WAS so it works
    #hh_WAS["CASEW5"] = hh_WAS["CASEW5"].replace(r"^\s*$", pd.NA, regex=True)
    #hh_WAS_empty = hh_WAS["CASEW5"].isna().sum()
    #hh_WAS = hh_WAS.dropna(subset=["CASEW5"])
    #hh_WAS_empty = hh_WAS["CASEW5"].isna().sum()

    #count and record lenth of dataframes
    hh_WAS_length = len(hh_WAS)
    intermediates["hh WAS length after duplicate drop"] = hh_WAS_length

    # Merge household data to person data
    person_situ_df = person_WAS.merge( hh_WAS[[hh_ID_var, hh_type_var, hh_tenure_var]], on=hh_ID_var, how="left"
    )


    # Count rows with no match (i.e. where merged columns are NaN)
    number_of_person_ids_without_hh_match = person_situ_df[hh_type_var].isna().sum()
    intermediates["number_of_person_ids_without_hh_match"] = number_of_person_ids_without_hh_match

    person_situ_df_length = len(person_situ_df)
    intermediates["person situ df original length"] = person_situ_df_length

    ### 4 ###
    # Filter to renters or those living rent-free
    #c(`Not asked / applicable` = -9, `Don't know/ Refusal` = -8, `Owned outright` = 1, `Buying with help of mortgage / loan` = 2, `Part rent part mortgage (shared ownership)` = 3, Rented = 4, `Live here rent-free or Squatting` = 5)
    tenure_conditions = [4, 5] if survey_year == 2021 else [4, 5, 6]
    person_situ_df = person_situ_df[person_situ_df[hh_tenure_var].isin(tenure_conditions)]

    person_situ_df_length = len(person_situ_df)
    intermediates["person situ df length after filtering for renters and those living rent free"] = person_situ_df_length

    # Filter to 20-44 year-olds
    person_situ_df[age_group_var] = pd.to_numeric(person_situ_df[age_group_var], errors='coerce')
    #c(`Not Routed` = -9, `Don t know` = -8, `0-4` = 1, `5-9` = 2, `10-14` = 3, `15-19` = 4, `20-24` = 5, `25-29` = 6, `30-34` = 7, `35-39` = 8, `40-44` = 9, `45-49` = 10, `50-54` = 11, `55-59` = 12, `60-64` = 13, `65-69` = 14, `70-74` = 15, `75-79` = 16, `80+` = 17)
    person_situ_df = person_situ_df[person_situ_df[age_group_var].between(5, 9)]
    person_situ_df_length = len(person_situ_df)
    intermediates["person situ df length after filtering for 20 to 44 year olds"] = person_situ_df_length

    # Remove dependent children
    person_situ_df = person_situ_df[person_situ_df[dpndt_chld_var] != 1]
    person_situ_df_length = len(person_situ_df)
    intermediates["person situ df length after dropping dependent children"] = person_situ_df_length

    # Remove those who own other properties
    person_situ_df = person_situ_df[person_situ_df[own_property] != 1]
    person_situ_df_length = len(person_situ_df)
    intermediates["person situ df length after dropping those who already own a property"] = person_situ_df_length

    # Create household type groupings (1 = single, 2 = couple, 3 = other)
    HHold_type_map_dict = {1: 1, 2: 1, 3: 2, 4: 2, 5: 2, 6: 2, 7: 2, 8: 1, 9: 1, 10: 3}
    person_situ_df["HHTYPE"] = person_situ_df[hh_type_var].map(HHold_type_map_dict)

    #select singles for this function
    person_situ_df = person_situ_df[person_situ_df["HHTYPE"] == 1]
    person_situ_df_length = len(person_situ_df)
    intermediates["person situ df length filtered for singles"] = person_situ_df_length

    # Create new variable that is total wealth minus pension wealth
    person_situ_df["AVAILABLE_WEALTH"] = person_situ_df[person_ttl_wlth_var] - person_situ_df[pension_wealth]

    average_available_wealth= person_situ_df["AVAILABLE_WEALTH"].mean()
    intermediates["average_available_wealth"] = average_available_wealth

    #create region code to region name map
    region_mapping = {
        -9: "Not asked / applicable",
        1: "North East",
        2: "North West",
        4: "Yorkshire and The Humber",
        5: "East Midlands",
        6: "West Midlands",
        7: "East of England",
        8: "London",
        9: "South East",
        10: "South West",
        11: "Wales",
        12: "Scotland"
    }
    person_situ_df[region_var] = person_situ_df[region_var].map(region_mapping)

    #rename region_var
    person_situ_df = person_situ_df.rename(columns={region_var: "REGION"})

    #rename income variable
    person_situ_df = person_situ_df.rename(columns={person_ttl_pay_var: "AVAILABLE INCOME"})

    average_available_income= person_situ_df["AVAILABLE INCOME"].mean()
    intermediates["average_person_total_pay_or_available_income"] = average_available_income

    ### 5 ###
    #Add a column that is the average FTB house price for a respondants region for the year of the survey
    avg_first_time_hs_df = pd.read_csv("first_time_buyer_average_price_2026.csv")
    avg_first_time_hs_df = avg_first_time_hs_df.melt(id_vars=['Year'], var_name='REGION', value_name='AVERAGE_FTH_PRICE')

   # Use real data for target year if available, otherwise use survey year
    avg_first_time_hs_df_target = avg_first_time_hs_df[avg_first_time_hs_df["Year"] == year]

    if len(avg_first_time_hs_df_target) > 0:
        print(f"Step 5: Using real house price data for {year}")
        avg_first_time_hs_df_merge = avg_first_time_hs_df_target[["REGION", "AVERAGE_FTH_PRICE"]].copy()
        avg_first_time_hs_df_merge["AVERAGE_FTH_PRICE"] = avg_first_time_hs_df_merge["AVERAGE_FTH_PRICE"].astype(str).str.replace(',', '').astype(float)
    else:
        print(f"Step 5: No real data for {year}, using survey year {survey_year} prices")
        avg_first_time_hs_df_merge = avg_first_time_hs_df[avg_first_time_hs_df["Year"] == survey_year][["REGION", "AVERAGE_FTH_PRICE"]].copy()
        avg_first_time_hs_df_merge["AVERAGE_FTH_PRICE"] = avg_first_time_hs_df_merge["AVERAGE_FTH_PRICE"].astype(str).str.replace(',', '').astype(float)

    person_situ_df = person_situ_df.merge(avg_first_time_hs_df_merge, on="REGION", how='left')

    print("STEP 5 DONE")
    print("Columns:", person_situ_df.columns.tolist())
    print("AVERAGE_FTH_PRICE NaNs:", person_situ_df["AVERAGE_FTH_PRICE"].isna().sum())
    print("Regions in person_situ_df:", person_situ_df["REGION"].unique())
    print("Regions in avg_first_time_hs_df_merge:", avg_first_time_hs_df_merge["REGION"].unique())


    ### 6 ####
    #do analysis of FTB prices and aaffordability - this doesn't use WAS
    # Use real house price data for target year if available, otherwise inflate from survey year
    avg_first_time_hs_df_target = avg_first_time_hs_df[avg_first_time_hs_df["Year"] == year]

    if len(avg_first_time_hs_df_target) > 0:
        # Real data exists for target year - use it directly
        print(f"Using real house price data for {year}")
        analaysis_of_ftb_situtaion = avg_first_time_hs_df_target[["REGION", "AVERAGE_FTH_PRICE"]].copy()
        analaysis_of_ftb_situtaion["AVERAGE_FTH_PRICE"] = analaysis_of_ftb_situtaion["AVERAGE_FTH_PRICE"].astype(str).str.replace(',', '').astype(float)
    else:
        # No real data - inflate from survey year
        print(f"No real data for {year}, inflating from {survey_year}")
        analaysis_of_ftb_situtaion = avg_first_time_hs_df_merge.copy()
        analaysis_of_ftb_situtaion["AVERAGE_FTH_PRICE"] = analaysis_of_ftb_situtaion["AVERAGE_FTH_PRICE"] * find_increase_from_year(survey_year, year, house_price_time_series)

    # Apply SDLT conditional on parameter
    if apply_sdlt:
        analaysis_of_ftb_situtaion["AVERAGE_FTH_PRICE_WITH_SDLT"] = np.where(
            analaysis_of_ftb_situtaion["AVERAGE_FTH_PRICE"] >= 300000,
            analaysis_of_ftb_situtaion["AVERAGE_FTH_PRICE"] + (analaysis_of_ftb_situtaion["AVERAGE_FTH_PRICE"] - 300000) * 0.05,
            analaysis_of_ftb_situtaion["AVERAGE_FTH_PRICE"]
        )
    else:
        analaysis_of_ftb_situtaion["AVERAGE_FTH_PRICE_WITH_SDLT"] = analaysis_of_ftb_situtaion["AVERAGE_FTH_PRICE"]

    analaysis_of_ftb_situtaion["AVERAGE_FTH_SDLT"] = analaysis_of_ftb_situtaion["AVERAGE_FTH_PRICE_WITH_SDLT"] - analaysis_of_ftb_situtaion["AVERAGE_FTH_PRICE"]
    analaysis_of_ftb_situtaion["wealth needed for desposit and costs"] = analaysis_of_ftb_situtaion["AVERAGE_FTH_PRICE"] * deposit_size + analaysis_of_ftb_situtaion["AVERAGE_FTH_SDLT"] + purchase_costs + wiggle_room
    analaysis_of_ftb_situtaion["average_FTH_price_minus_discount"] = analaysis_of_ftb_situtaion.apply(
        lambda row: (
            row['AVERAGE_FTH_PRICE'] * (1 - london_discount)
            if row['REGION'] == 'London'
            else (row['AVERAGE_FTH_PRICE'] * (1 - discount))
        ),
        axis=1
    )
    analaysis_of_ftb_situtaion['income_required'] = analaysis_of_ftb_situtaion.apply(
        lambda row: (
            (row['AVERAGE_FTH_PRICE'] * (1 - london_discount - deposit_size)) / income_multiplier
            if row['REGION'] == 'London'
            else (row['AVERAGE_FTH_PRICE'] * (1 - discount - deposit_size)) / income_multiplier
        ),
        axis=1
    )
    analaysis_of_ftb_situtaion['monthly_payments'] = npf.pmt(interest_rate/12, mortgage_term*12, analaysis_of_ftb_situtaion['income_required'] * income_multiplier, fv=0, when='end')

    analaysis_of_ftb_situtaion.to_csv("wealth_and_income_needed_and_repayments.csv", index=False)
    print(analaysis_of_ftb_situtaion)
    print("STEP 6 DONE")

    ### 7 ####

    # Create average FTB price including stamp duty variable
    if apply_sdlt:
        person_situ_df["AVERAGE_FTH_PRICE_WITH_SDLT"] = np.where(
            person_situ_df["AVERAGE_FTH_PRICE"] >= 300000,
            person_situ_df["AVERAGE_FTH_PRICE"] + (person_situ_df["AVERAGE_FTH_PRICE"] - 300000) * 0.05,
            person_situ_df["AVERAGE_FTH_PRICE"]
        )
    else:
        person_situ_df["AVERAGE_FTH_PRICE_WITH_SDLT"] = person_situ_df["AVERAGE_FTH_PRICE"]

    person_situ_df["AVERAGE_FTH_SDLT"] = person_situ_df["AVERAGE_FTH_PRICE_WITH_SDLT"] - person_situ_df["AVERAGE_FTH_PRICE"]

    #add to intermediatories
    average_FTB_price = person_situ_df["AVERAGE_FTH_PRICE"].mean()
    intermediates["average_FTB_price"] = average_FTB_price
    average_FTB_price_with_SDLT = person_situ_df["AVERAGE_FTH_PRICE_WITH_SDLT"].mean()
    intermediates["average_FTB_price_with_SDLT"] = average_FTB_price_with_SDLT


    #create net income variable
    person_situ_df["AVAILABLE INCOME"] = pd.to_numeric(person_situ_df["AVAILABLE INCOME"], errors='coerce')
    person_situ_df["NET INCOME"] = person_situ_df["AVAILABLE INCOME"] - person_situ_df["AVAILABLE INCOME"].apply(calculate_income_tax) - person_situ_df["AVAILABLE INCOME"].apply(calculate_national_insurance)



    #create a new column that has living costs as a proportion of gross income then a new column that has estimated living costs
    #define conditions
    conditions = [
        (person_situ_df["AVAILABLE INCOME"] > 0) & (person_situ_df["AVAILABLE INCOME"] <  12948),
        (person_situ_df["AVAILABLE INCOME"] >=  12948) & (person_situ_df["AVAILABLE INCOME"] <  19760),
        (person_situ_df["AVAILABLE INCOME"] >= 19760) & (person_situ_df["AVAILABLE INCOME"] < 26104),
        (person_situ_df["AVAILABLE INCOME"] > 26104) & (person_situ_df["AVAILABLE INCOME"] < 32604),
        (person_situ_df["AVAILABLE INCOME"] >= 32604) & (person_situ_df["AVAILABLE INCOME"] < 39936),
        (person_situ_df["AVAILABLE INCOME"] >= 39936) & (person_situ_df["AVAILABLE INCOME"] < 48516),
        (person_situ_df["AVAILABLE INCOME"] > 48516) & (person_situ_df["AVAILABLE INCOME"] < 58864),
        (person_situ_df["AVAILABLE INCOME"] >= 58864) & (person_situ_df["AVAILABLE INCOME"] < 73008),
        (person_situ_df["AVAILABLE INCOME"] >= 73008) & (person_situ_df["AVAILABLE INCOME"] < 97292),
        (person_situ_df["AVAILABLE INCOME"] >= 97292)
    ]

    # Define corresponding values
    values = [0.7, 0.36, 0.3, 0.27, 0.24, 0.21,0.18, 0.17, 0.14, 0.15]

    # Apply conditions to create the new column
    person_situ_df["living_costs_share"] = np.select(conditions, values, default=np.nan)  # Default NaN for other cases

    #create living cost estimate
    person_situ_df["living_costs"]= person_situ_df["living_costs_share"] * person_situ_df["AVAILABLE INCOME"]

    #print average living costs
    print("living cost average")
    print(person_situ_df["living_costs"].mean())

    #create net_income_minus_living_costs variable
    person_situ_df["NET INCOME MINUS LIVING COSTS"] = person_situ_df["NET INCOME"] - person_situ_df["living_costs"]

    average_net_income_minus_living_costs = person_situ_df["NET INCOME MINUS LIVING COSTS"].mean()
    intermediates["average_net_income_minus_living_costs"] = average_net_income_minus_living_costs

    average_net_income= person_situ_df["NET INCOME"].mean()
    intermediates["average_net_income"] = average_net_income


    #print average net income net living costs
    print("net income minus living costs average")
    print(person_situ_df["NET INCOME MINUS LIVING COSTS"].mean())

    print("STEP 7 DONE")

    ### 8 ###
    #inflate all variables to 2025

    person_situ_df["NET INCOME MINUS LIVING COSTS"] = person_situ_df["NET INCOME MINUS LIVING COSTS"]*find_increase_from_year(survey_year, year, income_time_series)
    person_situ_df["AVAILABLE_WEALTH"] = person_situ_df["AVAILABLE_WEALTH"]*find_increase_from_year(survey_year, year, wealth_time_series)
    person_situ_df["AVAILABLE INCOME"] = person_situ_df["AVAILABLE INCOME"]*find_increase_from_year(survey_year, year, income_time_series)

    # Only inflate house prices if we didn't already use real data for target year
    if len(avg_first_time_hs_df[avg_first_time_hs_df["Year"] == year]) == 0:
        person_situ_df["AVERAGE_FTH_PRICE"] = person_situ_df["AVERAGE_FTH_PRICE"] * find_increase_from_year(survey_year, year, house_price_time_series)
        person_situ_df["AVERAGE_FTH_PRICE_WITH_SDLT"] = person_situ_df["AVERAGE_FTH_PRICE_WITH_SDLT"] * find_increase_from_year(survey_year, year, house_price_time_series)
        person_situ_df["AVERAGE_FTH_SDLT"] = person_situ_df["AVERAGE_FTH_SDLT"] * find_increase_from_year(survey_year, year, house_price_time_series)

    print("Columns after Step 8:", person_situ_df.columns.tolist())

    print("STEP 8 DONE")

    ### 9 ###
    print("ABOUT TO CREATE Afford_house...")

    #calc min wealth they can have
    #find the number of rows in person_situ_df
    person_situ_df_length = len(person_situ_df)
    person_situ_df["minimum_wealth_required"]= person_situ_df["AVERAGE_FTH_PRICE"] * deposit_size + person_situ_df["AVERAGE_FTH_SDLT"] + purchase_costs + wiggle_room
    weighted_total_hh_analysed = person_situ_df[person_weight_var].sum()
    intermediates["weighted total singles analysed"] = weighted_total_hh_analysed


    #create 'afford deposit' variable (is deposit big enough)
    person_situ_df["Afford_deposit"] = np.where(person_situ_df["AVAILABLE_WEALTH"] >= person_situ_df["minimum_wealth_required"],1,0)

    person_situ_df["Afford_deposit"] = person_situ_df["Afford_deposit"].astype(int)
    total_weighted_can_afford_deposit = (person_situ_df["Afford_deposit"] * person_situ_df[person_weight_var]).sum()
    total_who_can_afford_deposit = total_weighted_can_afford_deposit/weighted_total_hh_analysed
    intermediates["weighted total singles who can afford deposit"] = total_weighted_can_afford_deposit
    intermediates["share of singles who can afford deposit"] = total_who_can_afford_deposit


    #create max_of_min_welath_required_and_actual_wealth" variable which is the maximum of "available_wealth" and "minimum_deposit_size"
    person_situ_df["max_of_min_wealth_required_and_actual_wealth"] = person_situ_df[["minimum_wealth_required", "AVAILABLE_WEALTH"]].max(axis=1)

    # Create 'afford house' variable which sees whether purchase can borrow enough and has enough deposit to reach purchase price
    person_situ_df["Afford_house"] = np.where(
        person_situ_df["REGION"] == "London",
        np.where(
            person_situ_df["AVAILABLE INCOME"] * income_multiplier >=
            person_situ_df["AVERAGE_FTH_PRICE"]*(1 - london_discount) + person_situ_df["AVERAGE_FTH_SDLT"] + purchase_costs + wiggle_room - person_situ_df["max_of_min_wealth_required_and_actual_wealth"], 1, 0
        ),
        np.where(
            person_situ_df["AVAILABLE INCOME"] * income_multiplier >=
            person_situ_df["AVERAGE_FTH_PRICE"]*(1 - discount) + person_situ_df["AVERAGE_FTH_SDLT"] + purchase_costs + wiggle_room - person_situ_df["max_of_min_wealth_required_and_actual_wealth"], 1, 0
        )
    )

    person_situ_df["Afford_house"] = person_situ_df["Afford_house"].astype(int)
    total_weighted_can_afford_house = (person_situ_df["Afford_house"] * person_situ_df[person_weight_var]).sum()
    share_who_can_afford_house = total_weighted_can_afford_house/weighted_total_hh_analysed
    intermediates["weighted total singles who can afford house"] = total_weighted_can_afford_house
    intermediates["share of singles who can afford house"] = share_who_can_afford_house


    # Create 'Afford_mortgage' dummy varible which approximates banks analysis of whether a purcahser can afford the monthly repayments
    person_situ_df["Afford_mortgage"] = np.where(
        person_situ_df["REGION"] == "London",
        person_situ_df["NET INCOME MINUS LIVING COSTS"] >= (
            npf.pmt(
                (interest_rate+0.03) / 12, mortgage_term * 12,
                -(person_situ_df["AVERAGE_FTH_PRICE"]*(1 - london_discount) + person_situ_df["AVERAGE_FTH_SDLT"] + purchase_costs + wiggle_room - person_situ_df["max_of_min_wealth_required_and_actual_wealth"]
                )
            ) * 12  # Convert to annual payment
        ),
        person_situ_df["NET INCOME MINUS LIVING COSTS"] >= (
            npf.pmt(
                (interest_rate+0.03) / 12, mortgage_term * 12,
                -(person_situ_df["AVERAGE_FTH_PRICE"]*(1 - discount) + person_situ_df["AVERAGE_FTH_SDLT"] + purchase_costs + wiggle_room - person_situ_df["max_of_min_wealth_required_and_actual_wealth"]
                )
            ) * 12  # Convert to annual payment
        )
    )





    person_situ_df["Afford_mortgage"] = person_situ_df["Afford_mortgage"].astype(int)
    total_weighted_can_afford_mortgage = (person_situ_df["Afford_mortgage"] * person_situ_df[person_weight_var]).sum()
    share_who_can_afford_mortgage = total_weighted_can_afford_mortgage/weighted_total_hh_analysed
    intermediates["weighted total singles who can afford mortgage"] = total_weighted_can_afford_mortgage
    intermediates["share of singles who can afford mortgage"] = share_who_can_afford_mortgage

    print("share who can afford mortgae")
    print(person_situ_df["Afford_mortgage"].mean())



    print("share who can afford deposit")
    print(person_situ_df["Afford_deposit"].mean())

    print("average wealth")
    print(person_situ_df["AVAILABLE_WEALTH"].mean())

    print("average FTB price")
    print(person_situ_df["AVERAGE_FTH_PRICE"].mean())

    print("average income")
    print(person_situ_df["AVAILABLE INCOME"].mean())


    # Create 'can_buy' variables which is true if they can afford the mortgae, the deposit and the house
    person_situ_df["can_buy"] = person_situ_df["Afford_mortgage"]*person_situ_df["Afford_deposit"]*person_situ_df["Afford_house"]

    person_situ_df["can_buy"] = person_situ_df["can_buy"].astype(int)
    total_weighted_can_buy = (person_situ_df["can_buy"] * person_situ_df[person_weight_var]).sum()
    share_who_can_buy = total_weighted_can_buy/weighted_total_hh_analysed
    intermediates["weighted total singles who can buy"] = total_weighted_can_buy
    intermediates["share of singles who can buy"] = share_who_can_buy

    print("share who can buy")
    print(person_situ_df["can_buy"].mean())

    print(person_situ_df.columns)

    available_income_by_region = person_situ_df.groupby("REGION")["AVAILABLE INCOME"].mean()
    print("avilable_income_by_region")
    print(available_income_by_region)

    available_wealth_by_region = person_situ_df.groupby("REGION")["AVAILABLE_WEALTH"].median()
    print("avilable_wealth_by_region")
    print(available_wealth_by_region)


    # Group by 'can_buy' and 'REGION' and sum weights
    singles_who_can_buy = person_situ_df.groupby(["can_buy", "REGION"])[person_weight_var].sum().reset_index()

    print(singles_who_can_buy.columns)

    # Calculate UK totals for each 'can_buy' group and concatenate to main df
    total_can_buy = singles_who_can_buy.groupby("can_buy")[person_weight_var].sum().reset_index()
    total_can_buy["REGION"] = "UK"

    #create a new row for "can_buy" == 1 that has for "total" the  sum of all the regions of England, so "North East","North West","Yorkshire and The Humber","East Midlands","West Midlands","East of England","London","South East","South West"
    singles_who_can_buy = pd.concat([singles_who_can_buy, total_can_buy], ignore_index=True)

    # Define the regions to include in the sum
    regions_to_sum = [
        "North East", "North West", "Yorkshire and The Humber",
        "East Midlands", "West Midlands", "East of England",
        "London", "South East", "South West"
    ]

    #Filter for relevant regions
    filtered_df = singles_who_can_buy[singles_who_can_buy["REGION"].isin(regions_to_sum)]

    #print column headings for dataframe filtered_df
    print(filtered_df.columns)

    #Group by "can buy" and sum "total"
    england_totals = filtered_df.groupby("can_buy")[person_weight_var].sum().reset_index()

    #Add "Region" column with the value "England"
    england_totals["REGION"] = "England"

    #Append the new aggregated rows to the original DataFrame
    singles_who_can_buy = pd.concat([singles_who_can_buy, england_totals], ignore_index=True)

    # Pivot the table so 'can_buy' values become columns
    pivoted_df = singles_who_can_buy.pivot(index="REGION", columns="can_buy", values=person_weight_var)

    # Rename columns for clarity (if needed)
    pivoted_df.columns = [f"can_buy_{col}" for col in pivoted_df.columns]

    # Reset index to keep 'REGION' as a normal column
    pivoted_df = pivoted_df.reset_index()

    #
    singles_who_can_buy=pivoted_df

    #dd on new columns with the parameter values in it

    #rename column "can_buy_1" of "singles_who_can_buy", "singles who can buy"

    singles_who_can_buy = singles_who_can_buy.rename(columns={"can_buy_1": "singles who can buy"})
    singles_who_can_buy = singles_who_can_buy.rename(columns={"can_buy_0": "singles who can't buy"})

    singles_who_can_buy["total singles"]= singles_who_can_buy["singles who can buy"] + singles_who_can_buy["singles who can't buy"]

    singles_who_can_buy["percentage of singles who can buy"] = singles_who_can_buy["singles who can buy"] / singles_who_can_buy["total singles"]

    #put the total couples column to the right of the couples who can buy column
    # Get the current column list
    columns = list(singles_who_can_buy.columns)

    # Ensure 'total_couples' exists in the DataFrame before proceeding
    if "total singles" in columns:
    # Remove 'total_couples' from its current position
      columns.remove("total singles")

    # Insert 'total_couples' at the 4th position (index 3)
    columns.insert(3, "total singles")

    # Reorder the DataFrame
    singles_who_can_buy = singles_who_can_buy[columns]

    # Ensure 'total_couples' exists in the DataFrame before proceeding
    if "percentage of singles who can buy" in columns:
    # Remove 'total_couples' from its current position
      columns.remove("percentage of singles who can buy")

    # Insert 'total_couples' at the 4th position (index 3)
    columns.insert(4, "percentage of singles who can buy")

    # Reorder the DataFrame
    singles_who_can_buy = singles_who_can_buy[columns]

    # Corrected print statement
    print(f"Population by Household Type (can_buy):")



    # Corrected print statement
    #print(f"Population by Household Type (can_buy):")
    #print(singles_who_can_buy)

    #Create a dynamically named variable
    #df_singles = f"singles_{survey_year}_{int(deposit_size * 100)}_{income_multiplier}_{int(discount * 100)}_{int(london_discount * 100)}"

    #Assign the final DataFrame with the dynamically created name
    #globals()[df_singles] = singles_who_can_buy.copy()

    #Print confirmation
    #print(f"Created DataFrame: {df_singles}")

    #save dictionary intermediates to scv file

    df = pd.DataFrame(list(intermediates.items()), columns=['Key', 'Value'])

    # Print to screen
    print(df)

    # Save to CSV
    df.to_csv('intermediates.csv', index=False)

    # *** ADD THIS BEFORE return ***
    import gc
    del person_WAS, hh_WAS
    gc.collect()
    # *** END OF ADDED BLOCK ***


    return singles_who_can_buy


#singles_who_can_buy(2015,2015, 0.05, 4, 0, 0,2500,0.04,25,0)

#(survey_year,deposit_size,income_multiplier,discount,london_discount,purchase_costs,interest_rate,mortgage_term)



5. Couples who are renting or living rent free

In [21]:
os.chdir(project_path)

def couples_who_can_buy(survey_year, year, deposit_size,income_multiplier,discount,london_discount,purchase_costs,interest_rate,mortgage_term, wiggle_room, breakup_percentage, apply_sdlt=True):
    year_round_lookup_dict = {2021: "round_8", 2019: "round_7", 2017: "round_6", 2015: "round_5"}

    ### 1 ###
    # Create empty dictionary to save intermediate values
    intermediates_couples = {}

    if survey_year not in year_round_lookup_dict:
        return "Data not available for this year"

    # Load the person and household level WAS data
    person_WAS = pd.read_csv(f"was_{year_round_lookup_dict[survey_year]}_person.tab", sep="\t")
    hh_WAS = pd.read_csv(f"was_{year_round_lookup_dict[survey_year]}_hhold.tab", sep="\t")

    print(f"Number of rows in person_WAS: {len(person_WAS)}")
    print(f"Number of rows in hh_WAS: {len(hh_WAS)}")

    # Load variable names
    variable_name_sheet = load_data_from_google_sheet(
        "https://docs.google.com/spreadsheets/d/1WHPnzAXqIiGfw2wricmdsWYnSKKMRrnMQEN1NQhdwWM/edit?usp=sharing", "Data"
    )
    variable_name_sheet = variable_name_sheet[variable_name_sheet["Year"].astype(int) == survey_year]
    variable_map = variable_name_sheet.set_index("Variable name")["Round variable name"].to_dict()

    # Retrieve and assign values
    person_ID_var = variable_map["Person_ID"]
    hh_ID_var = variable_map["Household_ID"]
    hh_type_var = variable_map["Household_type"]
    hh_tenure_var = variable_map["Tenure_type"]
    non_dpndt_chld_var = variable_map["Non_dependent_child_flag"]
    person_ttl_wlth_var = variable_map["Person_total_wealth"]
    dpndt_chld_var = variable_map["Dependent_child_flag"]
    person_ttl_pay_var = variable_map["Person_total_pay"]
    emplmnt_situ_var = variable_map["Employment_situation"]
    age_group_var = variable_map["Age_grouping"]
    person_weight_var = variable_map["Person_weight"]
    region_var = variable_map["Region"]
    own_property = variable_map["Own_property"]
    pension_wealth = variable_map["Pension_wealth"]

    # *** ADD THIS BLOCK HERE TO GET RID OF UNUSED COLUMNS ***
    cols_to_keep = [person_ID_var, hh_ID_var, hh_type_var, hh_tenure_var,
                    non_dpndt_chld_var, person_ttl_wlth_var, dpndt_chld_var,
                    person_ttl_pay_var, emplmnt_situ_var, age_group_var,
                    person_weight_var, region_var, own_property, pension_wealth]
    person_WAS = person_WAS[[c for c in cols_to_keep if c in person_WAS.columns]]
    hh_WAS = hh_WAS[[hh_ID_var, hh_type_var, hh_tenure_var]]
    # *** END OF ADDED BLOCK ***

    #count the number of unique values in the column hh_ID_var
    unique_hh_ids = hh_WAS[hh_ID_var].nunique()
    print(unique_hh_ids)

    #summerize the variable hh_ID_var in hh_WAS
    print(hh_WAS[hh_ID_var].value_counts())

    #hh_WAS["CASEW5"] = hh_WAS["CASEW5"].replace(r"^\s*$", pd.NA, regex=True)

    #hh_WAS_empty = hh_WAS["CASEW5"].isna().sum()
    #print(hh_WAS_empty)
    #hh_WAS = hh_WAS.dropna(subset=["CASEW5"])
    #hh_WAS_empty = hh_WAS["CASEW5"].isna().sum()
    #print(hh_WAS_empty)

    #print(f"Number of rows in hh_WAS: {len(hh_WAS)}")

    # Merge household data to person data
    person_situ_df = person_WAS.merge(
        hh_WAS[[hh_ID_var, hh_type_var, hh_tenure_var]], on=hh_ID_var, how="left"
    )

    print(f"Rows in person_situ_df: {len(person_situ_df)}")

    # Filter to renters or those living rent-fre
    #c(`Not asked / applicable` = -9, `Don't know/ Refusal` = -8, `Owned outright` = 1, `Buying with help of mortgage / loan` = 2, `Part rent part mortgage (shared ownership)` = 3, Rented = 4, `Live here rent-free or Squatting` = 5)
    tenure_conditions = [4, 5] if survey_year == 2021 else [4, 5, 6]
    person_situ_df = person_situ_df[person_situ_df[hh_tenure_var].isin(tenure_conditions)]
    print(f"Rows filtered to renters: {len(person_situ_df)}")

    # Filter to 20-44 year-olds
    person_situ_df[age_group_var] = pd.to_numeric(person_situ_df[age_group_var], errors='coerce')

    #c(`Not Routed` = -9, `Don t know` = -8, `0-4` = 1, `5-9` = 2, `10-14` = 3, `15-19` = 4, `20-24` = 5, `25-29` = 6, `30-34` = 7, `35-39` = 8, `40-44` = 9, `45-49` = 10, `50-54` = 11, `55-59` = 12, `60-64` = 13, `65-69` = 14, `70-74` = 15, `75-79` = 16, `80+` = 17)
    person_situ_df = person_situ_df[person_situ_df[age_group_var].between(5, 9)]
    print(f"Rows filtered to 20-44 years old: {len(person_situ_df)}")

    # Remove dependent children
    person_situ_df = person_situ_df[person_situ_df[dpndt_chld_var] != 1]
    print(f"Rows without dependent children: {len(person_situ_df)}")

    # Remove those who own other properties
    person_situ_df = person_situ_df[person_situ_df[own_property] != 1]
    print(f"Rows without own property: {len(person_situ_df)}")

    # Create household type groupings (1 = single, 2 = couple, 3 = other)
    HHold_type_map_dict = {1: 1, 2: 1, 3: 2, 4: 2, 5: 2, 6: 2, 7: 2, 8: 1, 9: 1, 10: 3}
    person_situ_df["HHTYPE"] = person_situ_df[hh_type_var].map(HHold_type_map_dict)

    print(f"Rows in each household type:\n{person_situ_df['HHTYPE'].value_counts()}")



    # Print weighted count of different household types
    person_situ_df[person_weight_var] = pd.to_numeric(person_situ_df[person_weight_var], errors='coerce')
    population_by_HHtype = person_situ_df.groupby("HHTYPE")[person_weight_var].sum().reset_index()

    # Corrected print statement
    print(f"Population by Household Type (HHTYPE):")
    print(population_by_HHtype)

    #filter to only couples
    person_situ_df = person_situ_df[person_situ_df["HHTYPE"] == 2]

    person_situ_df[person_ttl_wlth_var] = pd.to_numeric(person_situ_df[person_ttl_wlth_var], errors="coerce")
    person_situ_df[pension_wealth] = pd.to_numeric(person_situ_df[pension_wealth], errors="coerce")


    # Create new variable that is total wealth minus pension wealth
    person_situ_df["AVAILABLE_WEALTH"] = person_situ_df[person_ttl_wlth_var] - person_situ_df[pension_wealth]

    #create region mapping
    region_mapping = {
        -9: "Not asked / applicable",
        1: "North East",
        2: "North West",
        4: "Yorkshire and The Humber",
        5: "East Midlands",
        6: "West Midlands",
        7: "East of England",
        8: "London",
        9: "South East",
        10: "South West",
        11: "Wales",
        12: "Scotland"
    }
    person_situ_df[region_var] = person_situ_df[region_var].map(region_mapping)

    #rename region_var
    person_situ_df = person_situ_df.rename(columns={region_var: "REGION"})

    #rename income variable
    person_situ_df = person_situ_df.rename(columns={person_ttl_pay_var:"AVAILABLE INCOME" })

    person_situ_df = person_situ_df[[
        "AVAILABLE_WEALTH",
        "AVAILABLE INCOME",
        person_ID_var,
        hh_ID_var,
        "REGION",
        person_weight_var
        ]]

    person_situ_df["AVAILABLE INCOME"] = pd.to_numeric(person_situ_df["AVAILABLE INCOME"], errors="coerce")


    #create net income variable
    person_situ_df["NET INCOME"] = person_situ_df["AVAILABLE INCOME"] - person_situ_df["AVAILABLE INCOME"].apply(calculate_income_tax) - person_situ_df["AVAILABLE INCOME"].apply(calculate_national_insurance)

    mean_net_income = person_situ_df["NET INCOME"].mean
    mean_gross_income =  person_situ_df["AVAILABLE INCOME"].mean

    print(f"mean_net_income {mean_net_income}.")
    print(f"mean_gross_income {mean_gross_income}.")

    #create a new column that has living costs as a proportion of gross income then a new column that has estimated living costs
    #define conditions
    conditions = [
        (person_situ_df["AVAILABLE INCOME"] > 0) & (person_situ_df["AVAILABLE INCOME"] <  12948),
        (person_situ_df["AVAILABLE INCOME"] >=  12948) & (person_situ_df["AVAILABLE INCOME"] <  19760),
        (person_situ_df["AVAILABLE INCOME"] >= 19760) & (person_situ_df["AVAILABLE INCOME"] < 26104),
        (person_situ_df["AVAILABLE INCOME"] > 26104) & (person_situ_df["AVAILABLE INCOME"] < 32604),
        (person_situ_df["AVAILABLE INCOME"] >= 32604) & (person_situ_df["AVAILABLE INCOME"] < 39936),
        (person_situ_df["AVAILABLE INCOME"] >= 39936) & (person_situ_df["AVAILABLE INCOME"] < 48516),
        (person_situ_df["AVAILABLE INCOME"] > 48516) & (person_situ_df["AVAILABLE INCOME"] < 58864),
        (person_situ_df["AVAILABLE INCOME"] >= 58864) & (person_situ_df["AVAILABLE INCOME"] < 73008),
        (person_situ_df["AVAILABLE INCOME"] >= 73008) & (person_situ_df["AVAILABLE INCOME"] < 97292),
        (person_situ_df["AVAILABLE INCOME"] >= 97292)
    ]

    # Define corresponding values
    values = [0.7, 0.36, 0.3, 0.27, 0.24, 0.21,0.18, 0.17, 0.14, 0.15]

    # Apply conditions to create the new column
    person_situ_df["living_costs_share"] = np.select(conditions, values, default=np.nan)  # Default NaN for other cases

    #create living cost estimate
    person_situ_df["living_costs"]= person_situ_df["living_costs_share"] * person_situ_df["AVAILABLE INCOME"]

    #print average living costs
    print("living cost average")
    print(person_situ_df["living_costs"].mean())

    #create net_income_minus_living_costs variable
    person_situ_df["NET INCOME MINUS LIVING COSTS"] = person_situ_df["NET INCOME"] - person_situ_df["living_costs"]

    #print average net income net living costs
    print("net income minus living costs average")
    print(person_situ_df["NET INCOME MINUS LIVING COSTS"].mean())

    #create a couples_situ_df which is the person_situ_dataframe colapsed using groupby on hh_ID_var where "AVAILABLE_WEALTH","AVAILABLE INCOME" and person_weight_var are teh sum of the rows and "region" is the first entry

    # Create couples_situ_df by collapsing person_situ_df using groupby on hh_ID_var
    couples_situ_df = person_situ_df.groupby(hh_ID_var, as_index=False).agg({
        "AVAILABLE_WEALTH": "sum",
        "AVAILABLE INCOME": "sum",
        person_weight_var: "sum",
        "NET INCOME MINUS LIVING COSTS": "sum",
        "NET INCOME": "sum",
        "REGION": "first"
    })

    # Add a new variable that counts the number of rows grouped into each hh_ID_var
    couples_situ_df["numer_of filtered_people_in_hh"] = person_situ_df.groupby(hh_ID_var).size().reset_index(drop=True)

    #print counts for unique values in "numer_of filtered_people_in_hh"
    print(couples_situ_df["numer_of filtered_people_in_hh"].value_counts())

    #Reduce to hhs where both people satisfy the criteria filtered for so far
    couples_situ_df = couples_situ_df[couples_situ_df["numer_of filtered_people_in_hh"] == 2]

    # Create a new dataframe that calculates the mean of the relevent variables
    couples_average = couples_situ_df[[
        "AVAILABLE_WEALTH",
        "AVAILABLE INCOME",
    ]].mean().to_frame().T  # Correct

    # Display the new dataframe
    print(couples_average)

    #Add the average house price for the year to the dataframe
    avg_first_time_hs_df = pd.read_csv("first_time_buyer_average_price_2026.csv")

    # Transform price_df from wide to long format
    avg_first_time_hs_df = avg_first_time_hs_df.melt(id_vars=['Year'], var_name='REGION', value_name='AVERAGE_FTH_PRICE')

    # Use real data for target year if available, otherwise use survey year and inflate
    avg_first_time_hs_df_target = avg_first_time_hs_df[avg_first_time_hs_df["Year"] == year]

    if len(avg_first_time_hs_df_target) > 0:
        print(f"Using real house price data for {year}")
        avg_first_time_hs_df_merge = avg_first_time_hs_df_target[["REGION", "AVERAGE_FTH_PRICE"]].copy()
        avg_first_time_hs_df_merge["AVERAGE_FTH_PRICE"] = avg_first_time_hs_df_merge["AVERAGE_FTH_PRICE"].astype(str).str.replace(',', '').astype(float)
        use_real_house_prices = True
    else:
        print(f"No real data for {year}, inflating from {survey_year}")
        avg_first_time_hs_df_merge = avg_first_time_hs_df[avg_first_time_hs_df["Year"] == survey_year][["REGION", "AVERAGE_FTH_PRICE"]].copy()
        avg_first_time_hs_df_merge["AVERAGE_FTH_PRICE"] = avg_first_time_hs_df_merge["AVERAGE_FTH_PRICE"].astype(str).str.replace(',', '').astype(float)
        use_real_house_prices = False

    couples_situ_df = couples_situ_df.merge(avg_first_time_hs_df_merge, on="REGION", how='left')
    couples_situ_df["AVAILABLE INCOME"] = pd.to_numeric(couples_situ_df["AVAILABLE INCOME"], errors='coerce')

    # Create average FTB price including stamp duty variable
    if apply_sdlt:
        couples_situ_df["AVERAGE_FTH_PRICE_WITH_SDLT"] = np.where(
            couples_situ_df["AVERAGE_FTH_PRICE"] >= 300000,
            couples_situ_df["AVERAGE_FTH_PRICE"] + (couples_situ_df["AVERAGE_FTH_PRICE"] - 300000) * 0.05,
            couples_situ_df["AVERAGE_FTH_PRICE"]
        )
    else:
        couples_situ_df["AVERAGE_FTH_PRICE_WITH_SDLT"] = couples_situ_df["AVERAGE_FTH_PRICE"]

    couples_situ_df["AVERAGE_FTH_SDLT"] = couples_situ_df["AVERAGE_FTH_PRICE_WITH_SDLT"] - couples_situ_df["AVERAGE_FTH_PRICE"]

    # Always inflate income and wealth from survey year to target year
    couples_situ_df["NET INCOME MINUS LIVING COSTS"] = couples_situ_df["NET INCOME MINUS LIVING COSTS"] * find_increase_from_year(survey_year, year, income_time_series)
    couples_situ_df["AVAILABLE_WEALTH"] = couples_situ_df["AVAILABLE_WEALTH"] * find_increase_from_year(survey_year, year, wealth_time_series)
    couples_situ_df["AVAILABLE INCOME"] = couples_situ_df["AVAILABLE INCOME"] * find_increase_from_year(survey_year, year, income_time_series)

    # Only inflate house prices if we didn't already use real data for the target year
    if not use_real_house_prices:
        couples_situ_df["AVERAGE_FTH_PRICE"] = couples_situ_df["AVERAGE_FTH_PRICE"] * find_increase_from_year(survey_year, year, house_price_time_series)
        couples_situ_df["AVERAGE_FTH_PRICE_WITH_SDLT"] = couples_situ_df["AVERAGE_FTH_PRICE_WITH_SDLT"] * find_increase_from_year(survey_year, year, house_price_time_series)
        couples_situ_df["AVERAGE_FTH_SDLT"] = couples_situ_df["AVERAGE_FTH_SDLT"] * find_increase_from_year(survey_year, year, house_price_time_series)

    #calc min wealth they can have
    couples_situ_df["minimum_wealth_required"]= couples_situ_df["AVERAGE_FTH_PRICE"] * deposit_size + couples_situ_df["AVERAGE_FTH_SDLT"] + purchase_costs + wiggle_room


    #calculate total number of people analysed and add to dictionary
    weighted_total_hh_analysed = couples_situ_df[person_weight_var].sum()
    intermediates_couples["weighted total people in couples analysed"] = weighted_total_hh_analysed

    #create 'afford deposit' variable (is deposit big enough)
    couples_situ_df["Afford_deposit"] = np.where(couples_situ_df["AVAILABLE_WEALTH"] >= couples_situ_df["minimum_wealth_required"],1,0)

    #save to dictionary
    couples_situ_df["Afford_deposit"] = couples_situ_df["Afford_deposit"].astype(int)
    total_weighted_can_afford_deposit = (couples_situ_df["Afford_deposit"] * couples_situ_df[person_weight_var]).sum()*(1-breakup_percentage)
    total_who_can_afford_deposit = total_weighted_can_afford_deposit/weighted_total_hh_analysed
    intermediates_couples["weighted total people in couples who can afford deposit and stay together"] = total_weighted_can_afford_deposit
    intermediates_couples["share of people in couples who can afford deposit and stay together"] = total_who_can_afford_deposit

    #create max_of_min_welath_required_and_actual_wealth" variable which is the maximum of "available_wealth" and "minimum_deposit_size"
    couples_situ_df["max_of_min_wealth_required_and_actual_wealth"] = couples_situ_df[["minimum_wealth_required", "AVAILABLE_WEALTH"]].max(axis=1)

    # Create 'afford house' variable which sees whether purchase can borrow enough and has enough deposit to reach purchase price
    couples_situ_df["Afford_house"] = np.where(
        couples_situ_df["REGION"] == "London",
        np.where(
            couples_situ_df["AVAILABLE INCOME"] * income_multiplier >=
            couples_situ_df["AVERAGE_FTH_PRICE"]*(1 - london_discount) + couples_situ_df["AVERAGE_FTH_SDLT"] + purchase_costs + wiggle_room - couples_situ_df["max_of_min_wealth_required_and_actual_wealth"], 1, 0
        ),
        np.where(
            couples_situ_df["AVAILABLE INCOME"] * income_multiplier >=
            couples_situ_df["AVERAGE_FTH_PRICE"]*(1 - discount) + couples_situ_df["AVERAGE_FTH_SDLT"] + purchase_costs + wiggle_room - couples_situ_df["max_of_min_wealth_required_and_actual_wealth"], 1, 0
        )
    )


    #save to dictionary
    couples_situ_df["Afford_house"] = couples_situ_df["Afford_house"].astype(int)
    total_weighted_can_afford_house = (couples_situ_df["Afford_house"] * couples_situ_df[person_weight_var]).sum()*(1-breakup_percentage)
    share_who_can_afford_house = total_weighted_can_afford_house/weighted_total_hh_analysed
    intermediates_couples["weighted total people in couples who can afford house and stay together"] = total_weighted_can_afford_house
    intermediates_couples["share of people in couples who can afford house and stay together"] = share_who_can_afford_house


    # Create 'Afford_mortgage' dummy varible which approximates banks analysis of whether a purcahser can afford the monthly repayments
    couples_situ_df["Afford_mortgage"] = np.where(
        couples_situ_df["REGION"] == "London",
        couples_situ_df["NET INCOME MINUS LIVING COSTS"] >= (
            npf.pmt(
                (interest_rate+0.03) / 12, mortgage_term * 12,
                -(couples_situ_df["AVERAGE_FTH_PRICE"]*(1 - london_discount) + couples_situ_df["AVERAGE_FTH_SDLT"] + purchase_costs + wiggle_room - couples_situ_df["max_of_min_wealth_required_and_actual_wealth"]
                )
            ) * 12  # Convert to annual payment
        ),
        couples_situ_df["NET INCOME MINUS LIVING COSTS"] >= (
            npf.pmt(
                (interest_rate+0.03) / 12, mortgage_term * 12,
                -(couples_situ_df["AVERAGE_FTH_PRICE"]*(1 - discount) + couples_situ_df["AVERAGE_FTH_SDLT"] + purchase_costs + wiggle_room - couples_situ_df["max_of_min_wealth_required_and_actual_wealth"]
                )
            ) * 12  # Convert to annual payment
        )
    )


    #save to dictionary
    couples_situ_df["Afford_mortgage"] = couples_situ_df["Afford_mortgage"].astype(int)
    total_weighted_can_afford_mortgage = (couples_situ_df["Afford_mortgage"] * couples_situ_df[person_weight_var]).sum()*(1-breakup_percentage)
    share_who_can_afford_mortgage = total_weighted_can_afford_mortgage/weighted_total_hh_analysed
    intermediates_couples["weighted total people in couples who can afford mortgage and stay together"] = total_weighted_can_afford_mortgage
    intermediates_couples["share of people in couples who can afford mortgage and stay together"] = share_who_can_afford_mortgage

    print("share who can afford mortgae")
    print(couples_situ_df["Afford_mortgage"].mean())


    print("share who can afford deposit")
    print(couples_situ_df["Afford_deposit"].mean())

    print("average wealth")
    print(couples_situ_df["AVAILABLE_WEALTH"].mean())

    print("average FTB price")
    print(couples_situ_df["AVERAGE_FTH_PRICE"].mean())

    print("average income")
    print(couples_situ_df["AVAILABLE INCOME"].mean())

    print("share who can afford house and stay tother")
    print(couples_situ_df["Afford_house"].mean())

    # Create 'can_buy' variables which is true if they can afford the mortgae, the deposit and the house
    couples_situ_df["can_buy"] = couples_situ_df["Afford_mortgage"]*couples_situ_df["Afford_deposit"]*couples_situ_df["Afford_house"]


    # add to dictionary
    couples_situ_df["can_buy"] = couples_situ_df["can_buy"].astype(int)
    total_weighted_can_buy = (couples_situ_df["can_buy"] * couples_situ_df[person_weight_var]).sum()*(1-breakup_percentage)
    share_who_can_buy = total_weighted_can_buy/weighted_total_hh_analysed
    intermediates_couples["weighted total couples who can buy and stay together"] = total_weighted_can_buy
    intermediates_couples["share ofcouples who can buy and stay together"] = share_who_can_buy

    print("share who can buy")
    print(couples_situ_df["can_buy"].mean())
    print(couples_situ_df.columns)

    # Group by HHTYPE and sum person_weight_var
    # Group by 'can_buy' and 'REGION' and sum weights
    couples_situ_df = couples_situ_df.groupby(["can_buy", "REGION"])[person_weight_var].sum().reset_index()

    # Calculate UK for each 'can_buy' group (across all regions)
    total_can_buy = couples_situ_df.groupby("can_buy")[person_weight_var].sum().reset_index()
    total_can_buy["REGION"] = "UK"

    # Combine all data together
    couples_who_can_buy  = pd.concat([couples_situ_df, total_can_buy], ignore_index=True)

    #create a new row for "can_buy" == 1 that has for "total" the  sum of all the regions of England, so "North East","North West","Yorkshire and The Humber","East Midlands","West Midlands","East of England","London","South East","South West"
    #couples_who_can_buy = pd.concat([couples_who_can_buy, total_can_buy], ignore_index=True)

    # Define the regions to include in the sum
    regions_to_sum = [
        "North East", "North West", "Yorkshire and The Humber",
        "East Midlands", "West Midlands", "East of England",
        "London", "South East", "South West"
    ]

    #Filter for relevant regions
    filtered_df = couples_who_can_buy[couples_who_can_buy["REGION"].isin(regions_to_sum)]

    #print column headings for dataframe filtered_df
    print(filtered_df.columns)

    #Group by "can buy" and sum "total"
    england_totals = filtered_df.groupby("can_buy")[person_weight_var].sum().reset_index()

    #Add "Region" column with the value "England"
    england_totals["REGION"] = "England"

    #Append the new aggregated rows to the original DataFrame
    couples_who_can_buy = pd.concat([couples_who_can_buy, england_totals], ignore_index=True)

    #drop duplicate rows
    #couples_who_can_buy = couples_who_can_buy.drop_duplicates()

    # Pivot the table so 'can_buy' values become columns
    pivoted_df = couples_who_can_buy.pivot(index="REGION", columns="can_buy", values=person_weight_var)

    # Rename columns for clarity (if needed)
    pivoted_df.columns = [f"can_buy_{col}" for col in pivoted_df.columns]

    # Reset index to keep 'REGION' as a normal column
    pivoted_df = pivoted_df.reset_index()

    couples_who_can_buy = pivoted_df

    #print column names
    print(couples_who_can_buy.columns)

    #rename column heading person_weight_var to 'Total'
    couples_who_can_buy = couples_who_can_buy.rename(columns={person_weight_var: "total"})

    #print column names
    print(couples_who_can_buy.columns)

    #rename column "can_buy_1" of "couples_who_can_buy", "couples who can buy"

    couples_who_can_buy = couples_who_can_buy.rename(columns={"can_buy_1": "people in couples who can buy"})
    couples_who_can_buy = couples_who_can_buy.rename(columns={"can_buy_0": "people in couples who can't buy"})
    couples_who_can_buy["people in couples who can buy"]=couples_who_can_buy["people in couples who can buy"]*(1-breakup_percentage)
    couples_who_can_buy["people in couples who can't buy"]=couples_who_can_buy["people in couples who can't buy"] + couples_who_can_buy["people in couples who can buy"]*(breakup_percentage)
    couples_who_can_buy["total people in couples"]= couples_who_can_buy["people in couples who can buy"] + couples_who_can_buy["people in couples who can't buy"]
    couples_who_can_buy["percentage of people in couples who can buy"] = couples_who_can_buy["people in couples who can buy"] / couples_who_can_buy["total people in couples"]



    #put the total couples column to the right of the couples who can buy column
    # Get the current column list
    columns = list(couples_who_can_buy.columns)

    # Ensure 'total_couples' exists in the DataFrame before proceeding
    if "total people in couples" in columns:
    # Remove 'total_couples' from its current position
      columns.remove("total people in couples")

    # Insert 'total_couples' at the 4th position (index 3)
    columns.insert(3, "total people in couples")

    # Reorder the DataFrame
    couples_who_can_buy = couples_who_can_buy[columns]

    # Ensure 'total_couples' exists in the DataFrame before proceeding
    if "percentage of people in couples who can buy" in columns:
    # Remove 'total_couples' from its current position
      columns.remove("percentage of people in couples who can buy")

    # Insert 'total_couples' at the 4th position (index 3)
    columns.insert(4, "percentage of people in couples who can buy")

    # Reorder the DataFrame
    couples_who_can_buy = couples_who_can_buy[columns]

    # Define new columns to add
    new_columns = {
        "survey_year": survey_year,
        "deposit_size": deposit_size,
        "income_multiplier": income_multiplier,
        "discount": discount,
        "london_discount": london_discount,
        "purchase_costs": purchase_costs,
        "interest_rate": interest_rate,
        "mortgage_term": mortgage_term,
        "wiggle_room": wiggle_room,
        "breakup_percentage": breakup_percentage,
        "apply_sdlt": apply_sdlt
    }

    # Add new columns to the DataFrame
    for col, value in new_columns.items():
        couples_who_can_buy[col] = value

    # Reorder DataFrame so new columns are at the front
    couples_who_can_buy = couples_who_can_buy[list(new_columns.keys()) + [col for col in couples_who_can_buy.columns if col not in new_columns]]

    # Corrected print statement
    print(f"Population by Household Type (can_buy):")

    #Create a dynamically named variable
    df_couples = f"couples_{survey_year}_{int(deposit_size * 100)}_{income_multiplier}_{int(discount * 100)}_{int(london_discount * 100)}"

    #Assign the final DataFrame with the dynamically created name
    globals()[ df_couples] = couples_who_can_buy.copy()

    #Print confirmation
    print(f"Created DataFrame: {df_couples}")

    #save dictionary intermediates to scv file

    df = pd.DataFrame(list(intermediates_couples.items()), columns=['Key', 'Value'])

    # Print to screen
    print(df)

    # Save to CSV
    df.to_csv('intermediates_couples.csv', index=False)

    # *** ADD THIS BEFORE return ***
    import gc
    del person_WAS, hh_WAS
    gc.collect()
    # *** END OF ADDED BLOCK ***

    return couples_who_can_buy

#couples_who_can_buy(2015, 2015, 0.05, 4, 0, 0,1000,0.04,25,3000,0.1)

#(survey_year,deposit_size,income_multiplier,discount,london_discount,purchase_costs,interest_rate,mortgage_term, wiggle_room, breakup_percentage):



6. Run and save the model - can vary for parameters

In [26]:
#save csv file in results folder
os.chdir(project_path)

##def couples_who_can_buy(survey_year, year, deposit_size,income_multiplier,discount,london_discount,purchase_costs,interest_rate,mortgage_term, wiggle_room, breakup_percentage, apply_sdlt)

def save_dataframe(survey_year,year,deposit_size,income_multiplier,discount,london_discount,purchase_costs,interest_rate,mortgage_term, wiggle_room, breakup_percentage, apply_sdlt):
    filename = f"surveyyear_{survey_year}-year_{year}-deposit_{deposit_size}-incomemultiplier_{income_multiplier}-discount_{discount}-londondiscount_{london_discount}-purchasecosts_{purchase_costs}-interestrate_{interest_rate}-mortgageterm_{mortgage_term}-wiggleroom_{wiggle_room}-breakuppercentage_{breakup_percentage}-sdlt_{apply_sdlt}.csv"

    # Save the DataFrame as a CSV file
    df_combined.to_csv(filename, index=False)

    print(f"DataFrame saved as: {filename}")


singles_df=singles_who_can_buy(2021,2025, 0.1, 4.5, 0, 0,2500,0.05,25,0,apply_sdlt=True)
couples_df=couples_who_can_buy(2021,2025, 0.1, 4.5, 0, 0,2500,0.05,25,0,0.25,apply_sdlt=True)

#merge on "Region" column
df_combined = pd.merge(couples_df, singles_df, on='REGION', how='inner')

# Moves countries to bottom
england_row = df_combined[df_combined["REGION"] == "England"]
other_rows = df_combined[df_combined["REGION"] != "England"]
df_combined = pd.concat([other_rows, england_row], ignore_index=True)

scotland_row = df_combined[df_combined["REGION"] == "Scotland"]
other_rows = df_combined[df_combined["REGION"] != "Scotland"]
df_combined = pd.concat([other_rows, scotland_row], ignore_index=True)

wales_row = df_combined[df_combined["REGION"] == "Wales"]
other_rows = df_combined[df_combined["REGION"] != "Wales"]
df_combined = pd.concat([other_rows, wales_row], ignore_index=True)

UK_row = df_combined[df_combined["REGION"] == "UK"]
other_rows = df_combined[df_combined["REGION"] != "UK"]
df_combined = pd.concat([other_rows, UK_row], ignore_index=True)

df_combined["people who can buy"] = df_combined["people in couples who can buy"] + df_combined["singles who can buy"]
df_combined["people who can't buy"] = df_combined["people in couples who can't buy"] + df_combined["singles who can't buy"]
df_combined["total people"]= df_combined["total people in couples"] + df_combined["total singles"]
df_combined["percentage of people who can buy"]= df_combined["people who can buy"] / df_combined["total people"]

df_combined["homes that can be bought"] = df_combined["people in couples who can buy"]/2 + df_combined["singles who can buy"]

os.chdir('results')
save_dataframe(2021,2025, 0.1, 4.5, 0, 0,2500,0.05,25,0,0.25,apply_sdlt=True)

df_combined



/tmp/ipykernel_4549/1122444503.py:19: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,24,25,26,27,28,29,270,272,273,568,605,1143,3806,3815,3816,3817,3818,3819,3820,3821,3822,3823,3824,3825) have mixed types. Specify dtype option on import or set low_memory=False.
  person_WAS = pd.read_csv(f"was_{year_round_lookup_dict[survey_year]}_person.tab", sep="\t")


Step 5: Using real house price data for 2025
STEP 5 DONE
Columns: ['PersonR8', 'CASER8', 'IsNDepR8', 'P_TotalWlthR8', 'IsDepR8', 'AVAILABLE INCOME', 'PSitR8', 'DVAge17R8', 'R8xs_nonproxy_wgt', 'REGION', 'PropDVOPrValR8', 'totalpenr8', 'HHoldTypeR8', 'ten1r8_i', 'HHTYPE', 'AVAILABLE_WEALTH', 'AVERAGE_FTH_PRICE']
AVERAGE_FTH_PRICE NaNs: 0
Regions in person_situ_df: ['South West' 'North East' 'East Midlands' 'Scotland' 'West Midlands'
 'North West' 'London' 'South East' 'Yorkshire and The Humber' 'Wales'
 'East of England']
Regions in avg_first_time_hs_df_merge: ['Northern Ireland' 'Scotland' 'Wales' 'South West' 'South East' 'London'
 'East of England' 'West Midlands' 'East Midlands'
 'Yorkshire and The Humber' 'North West' 'North East']
Using real house price data for 2025
                       REGION  AVERAGE_FTH_PRICE  AVERAGE_FTH_PRICE_WITH_SDLT  \
39           Northern Ireland           189000.0                     189000.0   
79                   Scotland           185000.0       

/tmp/ipykernel_4549/2519958467.py:14: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,24,25,26,27,28,29,270,272,273,568,605,1143,3806,3815,3816,3817,3818,3819,3820,3821,3822,3823,3824,3825) have mixed types. Specify dtype option on import or set low_memory=False.
  person_WAS = pd.read_csv(f"was_{year_round_lookup_dict[survey_year]}_person.tab", sep="\t")


Number of rows in person_WAS: 32271
Number of rows in hh_WAS: 15128
15128
CASER8
12397    1
9055     1
2832     1
6226     1
2915     1
        ..
5771     1
14215    1
6320     1
4019     1
4464     1
Name: count, Length: 15128, dtype: int64
Rows in person_situ_df: 32271
Rows filtered to renters: 6163
Rows filtered to 20-44 years old: 1846
Rows without dependent children: 1846
Rows without own property: 1846
Rows in each household type:
HHTYPE
2    992
1    613
3    241
Name: count, dtype: int64
Population by Household Type (HHTYPE):
   HHTYPE  R8xs_nonproxy_wgt
0       1       3.146415e+06
1       2       4.418864e+06
2       3       1.304859e+06
mean_net_income <bound method Series.mean of 120          0.0
244      22959.6
385      85850.4
396      44197.4
406      19719.6
          ...   
31875        0.0
31996        0.0
32069        0.0
32215    24183.6
32265    17991.6
Name: NET INCOME, Length: 992, dtype: float64>.
mean_gross_income <bound method Series.mean of 120           0


,survey_year,deposit_size,income_multiplier,discount,london_discount,purchase_costs,interest_rate,mortgage_term,wiggle_room,breakup_percentage,...,percentage of people in couples who can buy,singles who can't buy,singles who can buy,total singles,percentage of singles who can buy,people who can buy,people who can't buy,total people,percentage of people who can buy,homes that can be bought
0,2021,0.1,4.5,0,0,2500,0.05,25,0,0.25,...,0.115450,2.018421e+05,4945.689719,2.067878e+05,0.023917,3.534786e+04,4.347759e+05,4.701237e+05,0.075188,20146.776904
1,2021,0.1,4.5,0,0,2500,0.05,25,0,0.25,...,0.149081,1.852113e+05,11546.963046,1.967583e+05,0.058686,6.454926e+04,4.877367e+05,5.522859e+05,0.116877,38048.110771
2,2021,0.1,4.5,0,0,2500,0.05,25,0,0.25,...,0.076852,4.234669e+05,11852.205866,4.353191e+05,0.027226,4.835022e+04,8.618795e+05,9.102297e+05,0.053119,30101.210493
3,2021,0.1,4.5,0,0,2500,0.05,25,0,0.25,...,0.080343,1.523296e+05,2292.437019,1.546220e+05,0.014826,1.787438e+04,3.306901e+05,3.485645e+05,0.051280,10083.410478
4,2021,0.1,4.5,0,0,2500,0.05,25,0,0.25,...,0.263256,3.865262e+05,5084.133530,3.916103e+05,0.012983,1.025330e+05,6.592452e+05,7.617782e+05,0.134597,53808.576301
5,2021,0.1,4.5,0,0,2500,0.05,25,0,0.25,...,0.271535,3.444952e+05,21127.073674,3.656223e+05,0.057784,1.673307e+05,7.367247e+05,9.040554e+05,0.185089,94228.910942
6,2021,0.1,4.5,0,0,2500,0.05,25,0,0.25,...,0.375643,2.551273e+05,40153.575126,2.952809e+05,0.135984,1.507617e+05,4.389695e+05,5.897313e+05,0.255645,95457.651338
7,2021,0.1,4.5,0,0,2500,0.05,25,0,0.25,...,0.219097,2.948599e+05,9986.535591,3.048465e+05,0.032759,9.354420e+04,5.926755e+05,6.862197e+05,0.136318,51765.367011
8,2021,0.1,4.5,0,0,2500,0.05,25,0,0.25,...,0.297754,3.086612e+05,17967.336890,3.266286e+05,0.055008,1.128415e+05,5.324194e+05,6.452609e+05,0.174877,65404.404690
9,2021,0.1,4.5,0,0,2500,0.05,25,0,0.25,...,0.209409,2.552520e+06,124955.950461,2.677476e+06,0.046669,7.931329e+05,5.075117e+06,5.868249e+06,0.135157,459044.418928


In [18]:
df_combined

,survey_year,deposit_size,income_multiplier,discount,london_discount,purchase_costs,interest_rate,mortgage_term,wiggle_room,breakup_percentage,...,percentage of people in couples who can buy,singles who can't buy,singles who can buy,total singles,percentage of singles who can buy,people who can buy,people who can't buy,total people,percentage of people who can buy,homes that can be bought
0,2021,0.05,4,0,0,2500,0.05,25,0,0.25,...,0.108905,2.061402e+05,647.558853,2.067878e+05,0.003132,29341.603141,4.409245e+05,4.702661e+05,0.062394,14994.580997
1,2021,0.05,4,0,0,2500,0.05,25,0,0.25,...,0.104645,1.856288e+05,11129.406167,1.967583e+05,0.056564,48470.047490,5.051210e+05,5.535911e+05,0.087556,29799.726828
2,2021,0.05,4,0,0,2500,0.05,25,0,0.25,...,0.128193,4.234669e+05,11852.205866,4.353191e+05,0.027226,72474.836045,8.357445e+05,9.082193e+05,0.079799,42163.520956
3,2021,0.05,4,0,0,2500,0.05,25,0,0.25,...,0.331527,1.509514e+05,3670.670528,1.546220e+05,0.023740,66658.110178,2.779560e+05,3.446141e+05,0.193428,35164.390353
4,2021,0.05,4,0,0,2500,0.05,25,0,0.25,...,0.289461,3.865262e+05,5084.133530,3.916103e+05,0.012983,112004.683249,6.489843e+05,7.609889e+05,0.147183,58544.408389
5,2021,0.05,4,0,0,2500,0.05,25,0,0.25,...,0.196654,3.569729e+05,8649.370770,3.656223e+05,0.023657,115184.661511,7.921764e+05,9.073611e+05,0.126945,61917.016140
6,2021,0.05,4,0,0,2500,0.05,25,0,0.25,...,0.305096,2.738343e+05,21446.516783,2.952809e+05,0.072631,111797.235283,4.796222e+05,5.914194e+05,0.189032,66621.876033
7,2021,0.05,4,0,0,2500,0.05,25,0,0.25,...,0.205825,2.948599e+05,9986.535591,3.048465e+05,0.032759,88568.107580,5.980663e+05,6.866344e+05,0.128989,49277.321585
8,2021,0.05,4,0,0,2500,0.05,25,0,0.25,...,0.310198,3.157565e+05,10872.062843,3.266286e+05,0.033286,109611.262602,5.353275e+05,6.449388e+05,0.169956,60241.662722
9,2021,0.05,4,0,0,2500,0.05,25,0,0.25,...,0.210237,2.594137e+06,83338.460930,2.677476e+06,0.031126,754110.547078,5.113923e+06,5.868033e+06,0.128512,418724.504004
